# Unit 6 solutions: grip and the speed limit

These reference solutions take the dynamic bicycle apart by hand, measure the grip ceiling on a skidpad, turn it into a speed cap along a lap, and then race it: a spin-out read from the telemetry, the margin rule tested on the noisy car, braking in a corner, and a shorter curvature horizon.
Each answer is predicted first where the lesson gives a law (the tyre curve, the friction circle, v_max = √(a_lat / |κ|)), and then measured.
Every race is seeded, so every number below is reproducible, and executable asserts check each answer against an independent derivation or the measured record.
Every number is a number about racekit's models (`DynamicCar` and `carla_like`), not the truth.
Exercises 5, 6 and 8 race; Exercise 6 races ten times and takes about a minute.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import DynamicCar, KinematicCar, carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import run_race
from racekit.tracks import load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_06 = AGENTS / "unit-06.py"
workdir = Path(mkdtemp(prefix="roar-unit-06-sol-"))

DT = 0.05  # s: one control tick
G = 9.81  # m/s^2
C_DRAG, A_THROTTLE, A_BRAKE = 4.0e-4, 6.0, 8.0  # the longitudinal map (KinematicCar and DynamicCar() share it)
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on max |cross-track error|
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
DELTA_MAX = math.radians(70.0)  # the wheel angle at |steer| = 1


def spins(result):
    # How many of the race's respawns were spin-outs.
    return sum(cause == "spin" for cause in result.respawn_causes)


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # The book's clean rule from Unit 6 on: finished, 0 collisions, max |offset| at most 2 m.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET

## Exercise 1 - Slip angles and tyre forces by hand

**Prediction.** The loads split by the lever arms: F_zf = 1845 · 9.81 · 1.475 / 2.875 ≈ 9286 N and F_zr ≈ 8814 N (they add to m g = 18099 N).
At v_x = 15 m/s the front axle moves sideways at v_y + l_f r = −0.3 + 0.7 = 0.4 m/s, a direction of about +0.027 rad, so α_f ≈ 0.027 − 0.12 ≈ −0.093 rad; the rear moves at −0.3 − 0.74 = −1.04 m/s, α_r ≈ −0.069 rad.
Both slip angles are negative, so both forces push **left** (positive), and both are below the tyre's peak, tan(π / 3.2) / 10 ≈ 0.150 rad: the tyres are still on the rising part of the curve.
The front force has the longer lever arm and is the larger one, so the yaw acceleration should be positive: with r = +0.5 rad/s the car is turning in harder.
The cornering stiffness is the small-α slope: sin(C atan(B α)) ≈ C B α, so C_α = B C μ F_zf = 16 · 9286 ≈ 148 573 N/rad.

In [ ]:
M, I_Z, L_F, L_R = 1845.0, 2900.0, 1.40, 1.475
WHEELBASE = L_F + L_R
B_SHAPE, C_SHAPE = 10.0, 1.6


def slip_angles(vx, vy, r, delta, lf=1.40, lr=1.475):
    alpha_front = math.atan2(vy + lf * r, vx) - delta
    alpha_rear = math.atan2(vy - lr * r, vx)
    return alpha_front, alpha_rear


def tyre_force(alpha, mu, fz, b=10.0, c=1.6):
    return -mu * fz * math.sin(c * math.atan2(b * alpha, 1.0))


fz_front, fz_rear = M * G * L_R / WHEELBASE, M * G * L_F / WHEELBASE
VX, VY, R, DELTA = 15.0, -0.3, 0.5, 0.12
alpha_f, alpha_r = slip_angles(VX, VY, R, DELTA)
force_f, force_r = tyre_force(alpha_f, 1.0, fz_front), tyre_force(alpha_r, 1.0, fz_rear)
lateral_accel = (force_f * math.cos(DELTA) + force_r) / M
yaw_accel = (L_F * force_f * math.cos(DELTA) - L_R * force_r) / I_Z

alpha_peak = math.tan(math.pi / (2.0 * C_SHAPE)) / B_SHAPE  # C atan(B alpha) = pi / 2
grid = np.linspace(0.0, 0.5, 50001)
alpha_peak_numeric = float(grid[np.argmax(np.abs([tyre_force(alpha, 1.0, fz_front) for alpha in grid]))])
stiffness_front = B_SHAPE * C_SHAPE * 1.0 * fz_front
H = 1e-6
stiffness_front_numeric = (tyre_force(-H, 1.0, fz_front) - tyre_force(H, 1.0, fz_front)) / (2.0 * H)
past_peak = {"front": abs(alpha_f) > alpha_peak, "rear": abs(alpha_r) > alpha_peak}
front_moment, rear_moment = L_F * force_f * math.cos(DELTA), L_R * force_r
turning_in = (
    f"The yaw rate is growing (yaw acceleration {yaw_accel:+.3f} rad/s^2 on top of r = +0.5 rad/s): the front "
    f"axle wins, its moment {front_moment:.0f} N m against the rear's {rear_moment:.0f} N m, so the car is still turning in."
)

print(f"loads: front {fz_front:.1f} N, rear {fz_rear:.1f} N")
print(f"slip angles: front {alpha_f:+.5f} rad ({math.degrees(alpha_f):+.2f} deg), rear {alpha_r:+.5f} rad ({math.degrees(alpha_r):+.2f} deg)")
print(f"forces: front {force_f:.1f} N, rear {force_r:.1f} N")
print(f"lateral acceleration {lateral_accel:.4f} m/s^2, yaw acceleration {yaw_accel:+.4f} rad/s^2")
print(f"peak slip angle: {alpha_peak:.6f} rad analytically, {alpha_peak_numeric:.6f} rad on the grid")
print(f"front cornering stiffness: {stiffness_front:.1f} N/rad analytically, {stiffness_front_numeric:.1f} N/rad numerically")
print("past the peak:", past_peak)
print(turning_in)

# independent checks: DynamicCar's own loads, the force at the peak, and the signs
car = DynamicCar()
assert abs(car.fz_f - fz_front) < 1e-9 and abs(car.fz_r - fz_rear) < 1e-9
assert abs(fz_front + fz_rear - M * G) < 1e-9
assert abs(abs(tyre_force(alpha_peak, 1.0, fz_front)) - fz_front) < 1e-9
assert abs(alpha_peak_numeric - alpha_peak) <= 1e-5  # one grid step
assert abs(stiffness_front_numeric / stiffness_front - 1.0) < 1e-6
assert alpha_f < 0.0 and alpha_r < 0.0 and force_f > 0.0 and force_r > 0.0
assert past_peak == {"front": False, "rear": False}
assert yaw_accel > 0.0 and front_moment > rear_moment
assert abs(lateral_accel - 8.5923) < 1e-4 and abs(yaw_accel - 0.46043) < 1e-5

## Exercise 2 - Where the two models part

**Prediction.** A steer of −0.05 is a wheel angle of 3.5°, a kinematic circle of κ = tan(3.5°) / 2.875 ≈ 0.0213 1/m (R ≈ 47 m).
The lesson showed that this neutral-steer car keeps the kinematic **curvature** at every speed but loses **speed** with a fixed throttle, because the steered front tyre's force has a component against the motion.
The yaw rate is r = v κ, so the yaw-rate ratio should track the speed ratio, falling further below 1 the faster the car goes (more lateral force, more drag from −F_yf sin δ), while the curvature ratio stays near 1.
At 25 m/s the circle asks for 25² · 0.0213 ≈ 13.3 m/s², more than μg = 9.81: the tyres cannot give it, so the car should spin.

In [ ]:
STEER_2 = -0.05
SPEEDS_2 = (2.5, 5.0, 10.0, 15.0, 20.0, 25.0)
spun, yaw_rate_ratio, curvature_ratio, speed_ratio = {}, {}, {}, {}
for v in SPEEDS_2:
    dynamic, kinematic = DynamicCar(), KinematicCar(lag_ticks=1)
    dynamic.reset(0.0, 0.0, 0.0, v)
    kinematic.reset(0.0, 0.0, 0.0, v)
    hold = C_DRAG * v ** 2 / A_THROTTLE
    spun_here = False
    for _ in range(200):
        dynamic.advance(hold, STEER_2, 0.0, 0.0, 0.0, DT, 10)
        kinematic.advance(hold, STEER_2, 0.0, 0.0, 0.0, DT, 10)
        spun_here = spun_here or dynamic.spun_out()
    spun[v] = spun_here
    r_kinematic = kinematic.v * math.tan(-STEER_2 * DELTA_MAX) / 2.875
    if spun_here:
        yaw_rate_ratio[v], curvature_ratio[v], speed_ratio[v] = None, None, None
    else:
        yaw_rate_ratio[v] = dynamic.r / r_kinematic
        curvature_ratio[v] = (dynamic.r / dynamic.v) / (r_kinematic / kinematic.v)
        speed_ratio[v] = dynamic.v / v

agree_to_1_percent = max([v for v in SPEEDS_2 if yaw_rate_ratio[v] is not None and abs(yaw_rate_ratio[v] - 1.0) <= 0.01])
demanded_at_25 = 25.0 ** 2 * math.tan(-STEER_2 * DELTA_MAX) / 2.875
where_they_part = (
    "At 10-20 m/s the dynamic car follows the same circle (curvature ratio about 1) but slower, because the fixed "
    "throttle no longer balances the steered front tyre's drag (-F_yf sin delta), and r = v kappa falls with the speed; "
    f"at 25 m/s the circle demands {demanded_at_25:.1f} m/s^2, more than mu g = 9.81 m/s^2, so the tyres saturate and the car spins."
)

print(f"{'v':>5}{'spun':>7}{'r ratio':>10}{'kappa ratio':>13}{'v ratio':>10}")
for v in SPEEDS_2:
    if spun[v]:
        print(f"{v:5.1f}{str(spun[v]):>7}{'-':>10}{'-':>13}{'-':>10}")
    else:
        print(f"{v:5.1f}{str(spun[v]):>7}{yaw_rate_ratio[v]:10.4f}{curvature_ratio[v]:13.5f}{speed_ratio[v]:10.4f}")
print(f"agree to 1%: up to {agree_to_1_percent} m/s; demanded at 25 m/s: {demanded_at_25:.3f} m/s^2")
print(where_they_part)

assert spun == {2.5: False, 5.0: False, 10.0: False, 15.0: False, 20.0: False, 25.0: True}
assert agree_to_1_percent == 5.0
for v in SPEEDS_2[:-1]:
    assert abs(curvature_ratio[v] - 1.0) < 0.002  # the same path
    assert abs(yaw_rate_ratio[v] - speed_ratio[v]) < 0.002  # r = v kappa: the yaw rate falls with the speed
for slower, faster in zip(SPEEDS_2[:-2], SPEEDS_2[1:-1]):
    assert yaw_rate_ratio[faster] < yaw_rate_ratio[slower]
assert demanded_at_25 > G and abs(demanded_at_25 - 13.2962) < 1e-4

## Exercise 3 - The skidpad on a wet track

**Prediction.** The skidpad circle is κ₀ = tan(7°) / 2.875 ≈ 0.0427 1/m.
The friction circle caps |a_lat| at μg, so the peak should be just under μg at every μ (a little of the circle goes to the throttle that keeps the speed rising, as on the lesson's dry skidpad), and the spin should come near v = √(μg / κ₀): about 10.7, 12.7 and 15.2 m/s for μ = 0.5, 0.7 and 1.0.

In [ ]:
STEER_3 = -0.1
kappa_0 = math.tan(-STEER_3 * DELTA_MAX) / 2.875
MUS = (0.5, 0.7, 1.0)
peak_lat, peak_fraction, spin_speed, predicted_speed, runs = {}, {}, {}, {}, {}
for mu in MUS:
    car = DynamicCar(mu=mu)
    car.reset(0.0, 0.0, 0.0, 5.0)
    speeds, lats = [], []
    spin_speed[mu] = None
    for n in range(4000):
        goal = 5.0 + 0.2 * n * 0.05
        throttle = min(max(0.5 * (goal - car.v) + C_DRAG * car.v ** 2 / A_THROTTLE, 0.0), 1.0)
        vx, vy = car.vx, car.vy
        car.advance(throttle, STEER_3, 0.0, 0.0, 0.0, 0.05, 10)
        speeds = speeds + [car.v]
        lats = lats + [(car.vy - vy) / 0.05 + car.vx * car.r]
        if car.spun_out():
            spin_speed[mu] = car.v
            break
    runs[mu] = (np.array(speeds), np.abs(np.array(lats)))
    peak_lat[mu] = float(np.max(runs[mu][1]))
    peak_fraction[mu] = peak_lat[mu] / (mu * G)
    predicted_speed[mu] = math.sqrt(mu * G / kappa_0)

fig, ax = plt.subplots(figsize=(8, 4.5))
for mu, colour in zip(MUS, ("tab:blue", "tab:orange", "tab:green")):
    ax.plot(runs[mu][0], runs[mu][1], color=colour, label=f"|a_lat|, mu = {mu}")
    ax.axhline(mu * G, color=colour, linestyle="--", label=f"mu g = {mu * G:.2f} m/s^2")
ax.set_xlabel("v (m/s)")
ax.set_ylabel("|a_lat| (m/s^2)")
ax.set_title("skidpad at steer -0.1 on three grips: |a_lat| flattens just under mu g")
ax.legend()
plt.show()

scales_with_mu = min([peak_fraction[mu] for mu in MUS]) >= 0.95 and max([peak_fraction[mu] for mu in MUS]) <= 1.0
wet_track = (
    "Halving mu halves the lateral acceleration the tyres can give, and a corner needs v^2 kappa of it, so the fastest "
    "speed round the same corner falls by sqrt(2), to about 71% (not 50%): the speed enters the demand squared."
)

kappa_low = runs[1.0][1][runs[1.0][0] < 8.0] / runs[1.0][0][runs[1.0][0] < 8.0] ** 2
print(f"kappa_0 = {kappa_0:.5f} 1/m (the measured low-speed |a_lat| / v^2 is {float(np.median(kappa_low)):.5f})")
for mu in MUS:
    print(f"mu {mu}: peak |a_lat| {peak_lat[mu]:.3f} m/s^2 = {peak_fraction[mu]:.4f} mu g; "
          f"spun at {spin_speed[mu]:.2f} m/s, predicted {predicted_speed[mu]:.2f} m/s")
print(f"scales with mu: {scales_with_mu}")
print(wet_track)

assert scales_with_mu
for mu in MUS:
    assert spin_speed[mu] is not None
    assert 0.98 < peak_fraction[mu] < 1.0
    assert abs(spin_speed[mu] / predicted_speed[mu] - 1.0) < 0.05  # the friction circle predicts the spin
assert abs(float(np.median(kappa_low)) / kappa_0 - 1.0) < 0.01  # at low speed it holds the kinematic circle
assert spin_speed[0.5] < spin_speed[0.7] < spin_speed[1.0]

## Exercise 4 - The speed cap along a lap

**Prediction.** `monza-like` is mostly straights and long bends, so at A_LAT = 6 the cap should bind on only a small part of the lap: a bend binds only where |κ| > 6 / 20² = 0.015 1/m, a radius under about 67 m.
So the cap lap time should be only a little above the flat lap time.
On the `oval` (bends of R = 120 m, |κ| ≈ 1/120) the threshold is 20² / 120 ≈ 3.3 m/s².

In [ ]:
def menger_curvature(xs, ys):
    # Unit 2: the signed curvature at each waypoint from (i - 1, i, i + 1), cyclic; left positive.
    n = len(xs)
    before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
    ax_, ay_, cx_, cy_ = xs[before], ys[before], xs[after], ys[after]
    twice_area = (xs - ax_) * (cy_ - ay_) - (ys - ay_) * (cx_ - ax_)
    sides = np.hypot(xs - ax_, ys - ay_) * np.hypot(cx_ - xs, cy_ - ys) * np.hypot(cx_ - ax_, cy_ - ay_)
    return 2.0 * twice_area / sides


def curvature_ahead(kappa, horizon=25):
    # At each waypoint i, the sharpest |kappa| over waypoints i .. i + horizon - 1 (cyclic).
    n = len(kappa)
    window = (np.arange(n)[:, None] + np.arange(horizon)[None, :]) % n
    return np.max(np.abs(kappa[window]), axis=1)


def speed_cap(kappa_ahead, a_lat, target=20.0):
    return np.array([min(target, math.sqrt(a_lat / k)) if k > 1e-9 else target for k in kappa_ahead])


A_LAT = 6.0
monza = load_track("monza-like")
monza_kappa = menger_curvature(monza.x, monza.y)
monza_cap = speed_cap(curvature_ahead(monza_kappa), A_LAT)
capped_fraction = float(np.mean(monza_cap < 20.0))
slowest_index = int(np.argmin(monza_cap))
slowest_cap = float(monza_cap[slowest_index])
following = (np.arange(len(monza.x)) + 1) % len(monza.x)  # segment i runs from waypoint i to i + 1, cyclic
segment_length = np.hypot(monza.x[following] - monza.x, monza.y[following] - monza.y)
cap_lap_time = float(np.sum(segment_length / monza_cap))
flat_lap_time = float(np.sum(segment_length)) / 20.0

oval = load_track("oval")
oval_binding_a_lat = 20.0 ** 2 * float(np.max(np.abs(menger_curvature(oval.x, oval.y))))
why_oval = (
    f"Every A_LAT from 5 to 10 is above the oval's threshold of {oval_binding_a_lat:.2f} m/s^2, so the cap never "
    "drops below 20 m/s there and the cap agent races the oval exactly like the flat agent, whatever A_LAT is."
)

print(f"monza-like: {len(monza_cap)} waypoints; the cap is below 20 m/s on {capped_fraction * 100:.2f}% of them")
print(f"slowest cap {slowest_cap:.4f} m/s at waypoint {slowest_index} (R = {1.0 / float(np.max(np.abs(monza_kappa))):.1f} m)")
print(f"cap lap time {cap_lap_time:.3f} s against the flat {flat_lap_time:.3f} s (+{cap_lap_time - flat_lap_time:.3f} s)")
print(f"oval threshold {oval_binding_a_lat:.4f} m/s^2")
print(why_oval)

# independent checks
assert abs(slowest_cap - math.sqrt(A_LAT / float(np.max(np.abs(monza_kappa))))) < 1e-9
assert abs(np.sum(segment_length) - monza.length) < 1e-6
assert abs(capped_fraction - float(np.mean(curvature_ahead(monza_kappa) > A_LAT / 400.0))) < 1e-12
assert flat_lap_time < cap_lap_time < 1.05 * flat_lap_time
assert slowest_index == 597 and abs(slowest_cap - 12.2474) < 1e-4
assert abs(oval_binding_a_lat - 20.0 ** 2 / 120.0) < 0.01 and oval_binding_a_lat < 5.0

## Exercise 5 - Reading a spin-out from the telemetry

**Prediction.** The flat agent drives 20 m/s everywhere, and the lesson says it spins 39 times on `test-circuit` on `carla_like(noise=False)`.
Just before the first spin it should be near 20 m/s at the entry to a corner whose v²κ is above what the grip gives (the carla-like car's clean edge was 7 m/s²): a speed problem, so `over_by` should be clearly above 1.

In [ ]:
flat_agent = write_variant(UNIT_06, workdir / "unit-06-flat.py", {"SPEED_MODE": '"flat"'})
flat_result = run_race(flat_agent, "test-circuit", car=carla_like(noise=False), seed=0, max_seconds=1500.0)
spin_count = spins(flat_result)
first_spin_time = flat_result.respawn_times[0]
telemetry = flat_result.telemetry
row = int(np.nonzero((telemetry["t"] < first_spin_time) & (telemetry["held"] == 0))[0][-1])
spin_x, spin_y, spin_v = float(telemetry["x"][row]), float(telemetry["y"][row]), float(telemetry["v"][row])

circuit = load_track("test-circuit")
spin_waypoint = int(np.argmin(np.hypot(circuit.x - spin_x, circuit.y - spin_y)))
kappa_there = float(curvature_ahead(menger_curvature(circuit.x, circuit.y), 25)[spin_waypoint])
demanded_a_lat = spin_v ** 2 * kappa_there
allowed_speed = math.sqrt(7.0 / kappa_there)
over_by = spin_v / allowed_speed
diagnosis = (
    f"A speed problem: the car entered a corner of radius about {1.0 / kappa_there:.0f} m at {spin_v:.1f} m/s, "
    f"asking {demanded_a_lat:.1f} m/s^2 of tyres that hold about 7 (even mu g = 9.81 is less): over_by = {over_by:.2f}, "
    f"{over_by - 1:.0%} faster than the clean edge allows, and no steering law can make the tyres give more force."
)

print(f"{spin_count} spin-outs, {flat_result.collisions} collisions, finished {flat_result.finished}")
print(f"first respawn at {first_spin_time:.2f} s, cause {flat_result.respawn_causes[0]!r}")
print(f"last driving row before it: t = {float(telemetry['t'][row]):.2f} s, ({spin_x:.2f}, {spin_y:.2f}), v = {spin_v:.3f} m/s")
print(f"nearest waypoint {spin_waypoint}; curvature ahead {kappa_there:.5f} 1/m (R = {1.0 / kappa_there:.1f} m)")
print(f"demanded {demanded_a_lat:.3f} m/s^2; allowed at A_LAT 7: {allowed_speed:.3f} m/s; over by {over_by:.4f}")
print(diagnosis)

assert flat_result.respawn_causes[0] == "spin"
assert spin_count == 39 and not flat_result.finished  # the lesson's probe record
assert telemetry["t"][row] < first_spin_time and telemetry["held"][row] == 0
assert abs(math.hypot(circuit.x[spin_waypoint] - spin_x, circuit.y[spin_waypoint] - spin_y)
           - float(np.min(np.hypot(circuit.x - spin_x, circuit.y - spin_y)))) < 1e-12
assert demanded_a_lat > G and over_by > 1.2
assert abs(first_spin_time - 36.75) <= TICK

## Exercise 6 - Is the margin worth it?

**Prediction.** The lesson's record for A_LAT 6 on the noisy car is 5/5 clean, median 422.65 s, spread 0.30 s.
A_LAT 7 was clean in one deterministic race, but it is the **edge**: it sits on the last clean grid cell, and the throttle jitter moves the car's speed by a little every tick, so some seeds should tip it over into spin-outs.
A race still running at the 600 s cap is not finished, so it is not clean.
If even one seed is not clean, A_LAT 7 is not eligible and the rule ships 6.0, however fast 7's clean races are.

In [ ]:
alat_7 = write_variant(UNIT_06, workdir / "unit-06-alat-7.py", {"A_LAT": 7.0})
SEEDS = [0, 1, 2, 3, 4]
runs_6 = repeat_races(UNIT_06, "test-circuit", n=5, seeds=SEEDS, car_factory=lambda: carla_like(noise=True),
                      max_seconds=600.0)
runs_7 = repeat_races(alat_7, "test-circuit", n=5, seeds=SEEDS, car_factory=lambda: carla_like(noise=True),
                      max_seconds=600.0)
clean_6, clean_7 = [clean(result) for result in runs_6.results], [clean(result) for result in runs_7.results]
spins_6, spins_7 = [spins(result) for result in runs_6.results], [spins(result) for result in runs_7.results]
median_6, spread_6 = runs_6.median, runs_6.spread
clean_times_7 = [float(result.total_time) for result in runs_7.results if clean(result)]

medians = {6.0: runs_6.median, 7.0: runs_7.median}
eligible = [a_lat for a_lat, flags in ((6.0, clean_6), (7.0, clean_7)) if sum(flags) == len(flags)]
ship = min(eligible, key=lambda a_lat: medians[a_lat]) if eligible else None
unfinished_7 = sum(not result.finished for result in runs_7.results)
edge_on_noise = (
    f"One deterministic clean race at A_LAT 7 could not show that the edge is a knife edge: on the noisy car "
    f"{5 - sum(clean_7)} of five seeds are not clean ({sum(spins_7)} spin-outs, {unfinished_7} races still running at "
    "600 s), so a setting chosen at the edge is clean only by luck; the margin costs a few seconds against 7's clean "
    "race but buys five clean races out of five."
)

for name, flags, counts, runs in (("A_LAT 6", clean_6, spins_6, runs_6), ("A_LAT 7", clean_7, spins_7, runs_7)):
    times = ", ".join(f"{time:.2f}" for time in runs.times)
    print(f"{name}: clean {flags}, spin-outs {counts}, times [{times}] s")
print(f"A_LAT 6: median {median_6:.2f} s, spread {spread_6:.2f} s")
print(f"A_LAT 7 clean race times: {clean_times_7}; races still running at the 600 s cap: {unfinished_7}")
print(f"eligible: {eligible}; ship A_LAT = {ship}")
print(edge_on_noise)

assert clean_6 == [True] * 5 and spins_6 == [0] * 5
assert abs(median_6 - 422.65) <= TICK and abs(spread_6 - 0.30) <= TICK  # the lesson's probe record
assert sum(clean_7) < 5 and sum(spins_7) > 0
assert ship == 6.0
for time in clean_times_7:
    assert time < median_6  # 7's clean races are faster: the margin has a price

## Exercise 7 - Challenge: braking in the corner

**Prediction.** Before braking the car holds 14 m/s on the steer −0.1 circle (κ₀ ≈ 0.0427 1/m), so a_lat ≈ 14² · 0.0427 ≈ 8.4 m/s² (a little less, because the speed sits a touch under 14).
The friction circle leaves √(9.81² − (8b)²) for the corner while braking at 8b m/s²: 9.68, 9.27, 8.56, 7.43 and 5.68 m/s² for b = 0.2 … 1.0, so from b = 0.8 the corner asks for more than the circle has (`room_short_from`).
`spun_out` counts only above 5 m/s, so the large sideslip angles while the car comes to rest at the heavier brakes are not spins; the solution records the sideslip only above 5 m/s.
But `DynamicCar` splits the brake force by static load and has no load transfer: both axles lose the same share of their lateral grip together, so neither end lets go first.
The car should slide wide a little and slow down, not spin.

In [ ]:
BRAKES = (0.2, 0.4, 0.6, 0.8, 1.0)
STEER_7 = -0.1
lat_befores, spun_by_brake, peak_total, peak_fraction, room_for_lat = {}, {}, {}, {}, {}
min_curvature, max_sideslip = {}, {}
for b in BRAKES:
    car = DynamicCar()
    car.reset(0.0, 0.0, 0.0, 14.0)
    spun_by_brake[b] = False
    totals, curvatures, sideslips = [], [], []
    for n in range(480):
        braking = n >= 400
        throttle = 0.0 if braking else min(max(0.5 * (14.0 - car.v) + C_DRAG * car.v ** 2 / A_THROTTLE, 0.0), 1.0)
        brake = b if braking else 0.0
        vx, vy = car.vx, car.vy
        car.advance(throttle, STEER_7, brake, 0.0, 0.0, 0.05, 10)
        a_long = (car.vx - vx) / 0.05 - car.vy * car.r
        a_lat = (car.vy - vy) / 0.05 + car.vx * car.r
        if n == 399:
            lat_befores[b] = a_lat
        if braking:
            totals = totals + [math.hypot(a_long, a_lat)]
            if car.v > 5.0:  # the spin-out test only counts above 5 m/s
                curvatures = curvatures + [abs(car.r / car.v)]
                sideslips = sideslips + [abs(car.sideslip())]
            if car.spun_out():
                spun_by_brake[b] = True
                break
    peak_total[b] = max(totals)
    peak_fraction[b] = peak_total[b] / G
    room_for_lat[b] = math.sqrt(G ** 2 - (A_BRAKE * b) ** 2)
    min_curvature[b], max_sideslip[b] = min(curvatures), max(sideslips)

lat_before = lat_befores[BRAKES[0]]
short_of_room = [b for b in BRAKES if room_for_lat[b] < abs(lat_before)]
room_short_from = min(short_of_room) if short_of_room else None  # the first b whose circle leaves less than the corner used
spun_brakes = [b for b in BRAKES if spun_by_brake[b]]
smallest_spin_brake = min(spun_brakes) if spun_brakes else None
kappa_hold = math.tan(-STEER_7 * DELTA_MAX) / 2.875
why = (
    "It does not spin: braking takes lateral grip from both axles in proportion to their static loads, so the car "
    "slides a little wide (its path curvature dips) and slows down with its sideslip far from the 25 degree spin-out; the reason a "
    "real car behaves differently is the labelled limit 'no load transfer': a real car's braking unloads the rear "
    "tyres, the rear lets go first, and the car spins."
)

print(f"a_lat on the last hold tick: {lat_before:.4f} m/s^2 (the circle's v^2 kappa at 14 m/s is {14.0 ** 2 * kappa_hold:.3f})")
print(f"{'b':>4}{'spun':>7}{'peak |a|':>10}{'/ g':>8}{'room for a_lat':>16}{'min kappa':>11}{'max |beta| above 5 m/s':>24}")
for b in BRAKES:
    print(f"{b:4.1f}{str(spun_by_brake[b]):>7}{peak_total[b]:10.3f}{peak_fraction[b]:8.4f}{room_for_lat[b]:16.3f}"
          f"{min_curvature[b]:11.5f}{math.degrees(max_sideslip[b]):21.2f} deg")
print(f"the circle leaves less room than the corner used ({abs(lat_before):.3f} m/s^2) from b = {room_short_from}")
print(f"smallest brake that spun: {smallest_spin_brake}")
print(why)

for b in BRAKES:
    assert lat_befores[b] == lat_before  # the same hold every time
    assert not spun_by_brake[b]
    assert peak_total[b] <= G  # the friction circle holds
    assert max_sideslip[b] < math.radians(10.0)  # far from the 25 degree spin-out
assert smallest_spin_brake is None
assert abs(abs(lat_before) - 8.125) < 0.001 and abs(lat_before) < 14.0 ** 2 * kappa_hold
assert room_short_from == 0.8 and short_of_room == [0.8, 1.0]
assert room_for_lat[0.8] < abs(lat_before) < room_for_lat[0.6]
assert min_curvature[1.0] < 0.95 * kappa_hold  # under a full brake the car runs wide

## Exercise 8 - Challenge: how far ahead should the cap look?

**Prediction.** With a 10-waypoint (20 m) horizon the cap sees each corner later, so the car stays at 20 m/s longer and brakes later and harder: that should make it **faster** on the straights, but it reaches the corner closer to the cap and with less margin, so the max |offset| should grow.
At 20 m/s the full-brake distance to the tightest corner's cap (about 10.4 m/s at A_LAT 6) is (20² − 10.4²) / 16 ≈ 18 m, just inside 20 m, so the car can still slow in time; the edge could stay at 7 or move down.
Three single-seed races can only show this one seed on the deterministic car.

In [ ]:
HORIZON_25 = {6.0: 424.30, 7.0: 418.40}  # the lesson's horizon-25 row on carla_like(noise=False) (probe record)
horizon_10 = {}
for a_lat in (6.0, 7.0, 8.0):
    short = write_variant(UNIT_06, workdir / f"unit-06-h10-{a_lat}.py", {"CURVATURE_HORIZON": 10, "A_LAT": a_lat})
    result = run_race(short, "test-circuit", car=carla_like(noise=False), seed=0, max_seconds=1500.0)
    horizon_10[a_lat] = (clean(result), float(result.total_time), spins(result), max_offset(result))

clean_values = [a_lat for a_lat in (6.0, 7.0, 8.0) if horizon_10[a_lat][0]]
edge_10 = max(clean_values) if clean_values else None
same_edge = edge_10 == 7.0
faster_at_7 = horizon_10[7.0][0] and horizon_10[7.0][1] < 418.40
offset_cost_at_7 = horizon_10[7.0][3] - 1.13
horizon_verdict = (
    f"The 20 m horizon is faster (about {418.40 - horizon_10[7.0][1]:.1f} s at A_LAT 7) because the car holds 20 m/s "
    "until the corner is close and then brakes later and harder, but it costs margin: it reaches the corner nearer the "
    f"cap and drifts {offset_cost_at_7:.2f} m further off the line; three single-seed races on the deterministic car "
    "cannot tell whether that thinner margin stays clean on the noisy car or other seeds, or on other tracks."
)

for a_lat, (is_clean, time, count, offset) in horizon_10.items():
    print(f"horizon 10, A_LAT {a_lat}: clean {is_clean}, {time:.2f} s, {count} spin-outs, max |offset| {offset:.3f} m")
print(f"edge {edge_10} (same as horizon 25: {same_edge}); faster at 7: {faster_at_7}; offset cost at 7: {offset_cost_at_7:+.3f} m")
print(horizon_verdict)

assert edge_10 == 7.0 and same_edge
assert faster_at_7 and horizon_10[6.0][1] < HORIZON_25[6.0]  # faster at both clean settings
assert offset_cost_at_7 > 0.0
assert horizon_10[8.0][2] > 0 and not horizon_10[8.0][0]